# Models

TF-IDF settings first, with the baseline's logistic regression, then Naive Bayes and a linear SVM against it. Everything is scored on the grouped folds from the baselines notebook, and the test set isn't used. The notebook takes about 20 minutes to run, most of it building the two-word phrases for each fold.

In [1]:
import sys

sys.path.append("..")

import pandas as pd
from src.data import load_text, time_split
from src.models import (chosen, cross_validate, length_only, linear_svm, logistic, majority, naive_bayes,
                        score_folds, tfidf, vectorize_folds)

In [2]:
train, test = time_split(load_text())
X_train = train["text"]
y_train = train["disputed"]
opening = X_train.str.lower().str[:100]

In [3]:
def summary(scores):
    return {"roc_auc": scores["roc_auc"].mean(), "avg_precision": scores["avg_precision"].mean(),
            "roc_auc_std": scores["roc_auc"].std()}

## TF-IDF settings

Each setting is vectorized once per fold with `vectorize_folds()` and scored with the same logistic regression as the baseline. `features` is the size of the vocabulary in the first fold, and `roc_auc_std` is the spread between the five folds.

In [4]:
settings = {
    "words, stop words dropped (baseline)": dict(),
    "words": dict(stop_words=None),
    "words and two-word phrases": dict(stop_words=None, ngram_range=(1, 2)),
    "words and two-word phrases, stop words dropped": dict(ngram_range=(1, 2)),
    "words and two-word phrases, min_df 2": dict(stop_words=None, ngram_range=(1, 2), min_df=2),
    "words and two-word phrases, min_df 20": dict(stop_words=None, ngram_range=(1, 2), min_df=20),
}

rows = {}
for name, kwargs in settings.items():
    folds = vectorize_folds(tfidf(**kwargs), X_train, y_train, groups=opening)
    rows[name] = {"features": folds[0][0].shape[1], **summary(score_folds(logistic(), folds))}
tfidf_table = pd.DataFrame(rows).T
tfidf_table.round(4)

,features,roc_auc,avg_precision,roc_auc_std
"words, stop words dropped (baseline)",15684.0,0.6285,0.3432,0.0043
words,15992.0,0.6334,0.3472,0.0042
words and two-word phrases,243473.0,0.6442,0.3652,0.0057
"words and two-word phrases, stop words dropped",208716.0,0.6397,0.3594,0.0063
"words and two-word phrases, min_df 2",563822.0,0.6457,0.3664,0.0064
"words and two-word phrases, min_df 20",81373.0,0.6405,0.3600,0.0070


Leaving the stop words in adds 0.005 ROC-AUC with single words and about the same with two-word phrases (0.6442 against 0.6397), so the EDA's point about "did not" holds up. Two-word phrases add another 0.011 ROC-AUC and 0.018 average precision, for 15 times as many features. The minimum document frequency matters little: keeping phrases that are in only 2 narratives more than doubles the features for 0.0015 ROC-AUC, and asking for 20 loses 0.004. The classifiers below use words and two-word phrases with the stop words left in and a minimum of 5 narratives.

## Classifiers

Logistic regression with four values of C, Naive Bayes with three smoothing values and a linear SVM with four values of C, all on the same vectorized folds. The SVM has no probabilities, so it's scored on its decision values. Both measures only look at the order of the scores, so that's a fair comparison.

In [5]:
folds = vectorize_folds(tfidf(stop_words=None, ngram_range=(1, 2)), X_train, y_train, groups=opening)

classifiers = {
    "logistic, C 0.1": logistic(C=0.1),
    "logistic, C 0.3": logistic(C=0.3),
    "logistic, C 1": logistic(C=1),
    "logistic, C 3": logistic(C=3),
    "naive bayes, alpha 0.1": naive_bayes(alpha=0.1),
    "naive bayes, alpha 0.3": naive_bayes(alpha=0.3),
    "naive bayes, alpha 1": naive_bayes(alpha=1),
    "linear svm, C 0.01": linear_svm(C=0.01),
    "linear svm, C 0.03": linear_svm(C=0.03),
    "linear svm, C 0.1": linear_svm(C=0.1),
    "linear svm, C 1": linear_svm(C=1),
}

classifier_table = pd.DataFrame({name: summary(score_folds(model, folds)) for name, model in classifiers.items()}).T
classifier_table.round(4)

,roc_auc,avg_precision,roc_auc_std
"logistic, C 0.1",0.6463,0.3538,0.0045
"logistic, C 0.3",0.6499,0.3640,0.0056
"logistic, C 1",0.6442,0.3652,0.0057
"logistic, C 3",0.6334,0.3560,0.0073
"naive bayes, alpha 0.1",0.6284,0.3543,0.0049
"naive bayes, alpha 0.3",0.6139,0.3431,0.0049
"naive bayes, alpha 1",0.5569,0.2975,0.0052
"linear svm, C 0.01",0.6447,0.3510,0.0043
"linear svm, C 0.03",0.6503,0.3625,0.0055
"linear svm, C 0.1",0.6463,0.3649,0.0066


Logistic regression at C 0.3 and the SVM at C 0.03 are level at 0.650 ROC-AUC, and the gap between them is far smaller than the spread between folds (about 0.006). Both get worse with less regularisation, down to 0.633 and 0.616 at the highest C. Naive Bayes is behind at every setting. At the default alpha of 1 it drops to 0.557, below the length-only baseline, and 0.1 brings it to 0.628.

## The chosen model

I'm going with logistic regression at C 0.3. The SVM is level with it, but logistic regression gives probabilities, which the decision step needs, and its weights are easy to read in the explain step. C 0.3 has the better ROC-AUC (0.650 against 0.644 at C 1) for about 0.001 less average precision. `chosen()` in src/models.py is that model as a pipeline, and it's scored below the same way as the baselines.

In [6]:
oof_chosen, scores_chosen = cross_validate(chosen(), X_train, y_train, groups=opening)
scores_chosen.round(4)

,roc_auc,avg_precision
0,0.6525,0.3650
1,0.6494,0.3632
2,0.6514,0.3714
3,0.6407,0.3536
4,0.6557,0.3669


In [7]:
scores_chosen.mean()

roc_auc          0.649935
avg_precision    0.364023
dtype: float64

## Results so far

In [8]:
_, scores_majority = cross_validate(majority(), X_train, y_train, groups=opening)
_, scores_length = cross_validate(length_only(), X_train, y_train, groups=opening)

results = pd.DataFrame({
    "majority": scores_majority.mean(),
    "length only": scores_length.mean(),
    "tfidf + logistic, baseline": tfidf_table.iloc[0][["roc_auc", "avg_precision"]],
    "tfidf + logistic, chosen": scores_chosen.mean(),
}).T
results.astype(float).round(3)

,roc_auc,avg_precision
majority,0.500,0.232
length only,0.567,0.276
"tfidf + logistic, baseline",0.629,0.343
"tfidf + logistic, chosen",0.650,0.364


The chosen model gets a ROC-AUC of 0.650 and an average precision of 0.364, with the folds between 0.641 and 0.656. That's 0.021 more than the baseline on both, about four times the spread between folds. Its ranked list is about 1.6 times cleaner than picking at random (0.364 against 0.232).